# Experiment B: delay between OFAC listing and on-chain blacklisting (USDC and USDT)

Fetches the on-chain blacklist events of Circle (USDC) and Tether (USDT) and keeps only the events for the Ethereum-format addresses that OFAC listed in 2019-2023, exactly as fixed in the plan (docs/EXPLORATORY_PLAN.md, section B, tag v1.6-exploratory-plan). It does **not** analyse anything: the analysis runs later on GitHub from the committed data.

**How to use:** menu *Runtime, Run all*. When asked, paste a **new** Etherscan API key (the box is hidden; the key is never printed or saved). It takes a few minutes. At the end `exploratory_B_data.zip` downloads; send it back, then delete the key on etherscan.io/apidashboard.


In [ ]:
# Cell 2: functions (nothing to edit)
from __future__ import annotations

import csv, datetime as dt, hashlib, json, os, subprocess, sys, time, zipfile
from pathlib import Path

REPO_URL = "https://github.com/FahdSaif/RWA-Asset-tokenization.git"
REPO_REF = "v1.6-exploratory-plan"                      # the tag that fixes the plan, the code and the OFAC files used
CHAINID, OFFSET, MAX_PAGES, START_BLOCK = 1, 1000, 10, 6_900_000   # block 6.9M is about December 2018
BASE = "https://api.etherscan.io/v2/api"
TOKENS = {                                              # fixed in docs/EXPLORATORY_PLAN.md, section B
    "USDC": ("0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48", "0xffa4e6181777692565cf28528fc88fd1516ea86b56da075235fa575af6a4b855"),   # Blacklisted(address)
    "USDT": ("0xdAC17F958D2ee523a2206206994597C13D831ec7", "0x42e160154868087d6bfdc0ca23d96a1c1cfa32f1b72ba9ba27b69b98a0d819dc"),   # AddedBlackList(address)
}
OUT = Path("exploratory_B")


def log_url(address: str, topic0: str, from_block: int, page: int, key: str) -> str:
    return (f"{BASE}?chainid={CHAINID}&module=logs&action=getLogs&address={address}&topic0={topic0}"
            f"&fromBlock={from_block}&toBlock=latest&page={page}&offset={OFFSET}&apikey={key}")


def decode_address(topics: list[str], data: str) -> str:
    """The blacklisted address: topic 1 if the event indexes it, otherwise the last 20 bytes of the data field."""
    return "0x" + (topics[1][-40:] if len(topics) >= 2 else data[-40:]).lower()


def fetch_token_logs(get_json, key: str, address: str, topic0: str, pause: float = 1.0) -> list[dict]:
    """All logs of one event, oldest first, in windows of up to 10 pages (10,000 logs); a full window continues from its last block
    (inclusive; duplicates are dropped by transaction hash and log index). The key is never printed or saved."""
    events, seen, from_block = [], set(), START_BLOCK
    while True:
        window = []
        for page in range(1, MAX_PAGES + 1):
            d = get_json(log_url(address, topic0, from_block, page, key))
            if d.get("status") == "0" and "No records" in str(d.get("message")):
                break
            if d.get("status") != "1" or not isinstance(d.get("result"), list):
                raise RuntimeError(f"status={d.get('status')!r} message={d.get('message')!r} result={str(d.get('result'))[:120]!r}")
            window += d["result"]
            time.sleep(pause)
            if len(d["result"]) < OFFSET:
                break
        new = 0
        for r in window:
            k = (r["transactionHash"], r["logIndex"])
            if k in seen:
                continue
            seen.add(k)
            new += 1
            events.append(dict(address=decode_address(r["topics"], r["data"]), timestamp=int(r["timeStamp"], 16),
                               block=int(r["blockNumber"], 16), tx_hash=r["transactionHash"]))
        print(f"  window ending at block {max((int(r['blockNumber'], 16) for r in window), default=0)}: {len(window)} logs, {new} new")
        if len(window) < MAX_PAGES * OFFSET or new == 0:
            return events
        from_block = max(int(r["blockNumber"], 16) for r in window)


def sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def iso(ts: int) -> str:
    return dt.datetime.fromtimestamp(ts, dt.timezone.utc).strftime("%Y-%m-%d")


def build(all_events: dict[str, list[dict]], ofac: dict[str, str], repo_commit: str) -> dict:
    OUT.mkdir(exist_ok=True)
    rows, per_token = [], {}
    for tok, ev in all_events.items():
        (OUT / f"raw_{tok}.json").write_text(json.dumps(ev))
        sub = [e for e in ev if e["address"] in ofac]
        rows += [(tok, e["address"], e["timestamp"], e["block"], e["tx_hash"]) for e in sub]
        per_token[tok] = dict(events_total=len(ev), first_event=iso(min(e["timestamp"] for e in ev)) if ev else None,
                              last_event=iso(max(e["timestamp"] for e in ev)) if ev else None,
                              events_for_ofac_addresses=len(sub), ofac_addresses_blacklisted=len({e["address"] for e in sub}))
        if len(ev) < 50:
            print(f"WARNING: only {len(ev)} {tok} events were found: the event signature or contract may be wrong. Tell the author before using this data.")
    rows.sort(key=lambda r: (r[0], r[2], r[1]))
    with open(OUT / "blacklist_events_ofac_subset.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["token", "address", "timestamp", "block", "tx_hash"])
        w.writerows(rows)
    summary = dict(plan_tag=REPO_REF, repo_commit=repo_commit, retrieved_utc=dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                   start_block=START_BLOCK, tokens={t: dict(contract=TOKENS[t][0], topic0=TOKENS[t][1], **per_token[t]) for t in all_events},
                   ofac_addresses=len(ofac), subset_rows=len(rows),
                   sha256={p.name: sha256(p) for p in sorted(OUT.glob("*")) if p.name != "blacklist_summary.json"})
    (OUT / "blacklist_summary.json").write_text(json.dumps(summary, indent=1))
    with zipfile.ZipFile("exploratory_B_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.glob("*")):
            z.write(p, p.name)
    return summary


In [ ]:
# Cell 3: run
import getpass, requests
from google.colab import files

repo = os.environ.get("REPO_LOCAL", "repo")
if not os.path.exists(repo):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, repo], check=True)
commit = subprocess.run(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, f"{repo}/rwa-threatlab/src")
from rwa_threatlab.exploratory import ofac_addresses

OFAC = ofac_addresses.extract(f"{repo}/rwa-threatlab/docs/data/raw")
print(f"plan tag {REPO_REF}, commit {commit[:7]}: {len(OFAC)} OFAC addresses (0x format) listed 2019-2023")

key = getpass.getpass("Paste your Etherscan API key (hidden): ").strip()

def get_json(url):
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:          # never show the URL: it contains the key
        raise RuntimeError(f"network error: {type(e).__name__}") from None

all_events = {}
for tok, (addr, topic0) in TOKENS.items():
    print(f"fetching {tok} blacklist events ...")
    all_events[tok] = fetch_token_logs(get_json, key, addr, topic0)
    print(f"{tok}: {len(all_events[tok])} events in total")

summary = build(all_events, OFAC, commit)
print(json.dumps({k: v for k, v in summary.items() if k != "sha256"}, indent=1))
files.download("exploratory_B_data.zip")
